# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described with a Croissant schema available at the following URL:

 - https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load the dataset and review key metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# The Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Create a Dataset object
dataset = mlc.Dataset(croissant_url)

# Print top-level dataset metadata
md = dataset.metadata
print(f"Dataset name: {md.name}")
print(f"Version: {md.version}")
print(f"Description: {md.description}")
print(f"Date published: {md.datePublished}")
print(f"Keywords: {', '.join(md.keywords) if hasattr(md, 'keywords') else ''}")

## 2. Data Overview
Review available record sets and their fields. All references are by `@id` as defined in the Croissant schema.

We will enumerate all the record sets, their fields, and columns. Each entity is referenced using its `@id`.

In [ ]:
# List all record sets and their fields
record_sets = dataset.metadata.recordSet
record_set_ids = []
print("Available record sets:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', rs['@id'])}")
    record_set_ids.append(rs['@id'])
    # Enumerate fields for each record set
    if 'field' in rs:
        print("  Fields:")
        for field in rs['field']:
            print(f"    - @id: {field['@id']}, name: {field.get('name', field['@id'])}, dataType: {field.get('dataType', 'N/A')}")
    # Enumerate columns for each record set
    if 'column' in rs:
        print("  Columns:")
        for col in rs['column']:
            print(f"    - @id: {col['@id']}, name: {col.get('name', col['@id'])}, dataType: {col.get('dataType', 'N/A')}")


Below is a sample of the first few records from the primary record set.

> The code below loads and displays records for a selected record set. (Usually the main table of protein measures.)

**Replace `<main_record_set_id>` with the actual primary record set `@id` from the list above.**

In [ ]:
# Display a few example records from the main record set.
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"First records from record set: {main_record_set_id}")
    for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
        if i >= 3:
            break
        print(record)
else:
    print("No record sets found in the schema.")

## 3. Data Extraction
For analysis, load all record sets into pandas DataFrames. Each set is stored in the `dataframes` dictionary, keyed by the record set `@id`.

> Fields and columns are referenced by their `@id`s as above.

In [ ]:
dataframes = {}
for record_set_id in record_set_ids:
    # Load all records for this set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set @{record_set_id}: shape = {df.shape}")

# Preview the columns and first rows of the main record set
if main_record_set_id:
    print(f"Columns in main record set (@id={main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's perform EDA on the main record set.

Typical steps include filtering by a numeric field, normalizing, and grouping. **All field and groupings are referenced by their proper `@id`.**

> Update the field identifiers below after examining available fields in the overview above.


In [ ]:
if main_record_set_id:
    df = dataframes[main_record_set_id]

    # Example field IDs - please update based on field list above
    # Try to use a numeric field, e.g., abundance, MW, coverage, etc.
    numeric_field_id = None
    candidate_fields = []
    # Try to infer a numeric field
    for c in df.columns:
        # Heuristic: if column values are float or int for many values
        if pd.api.types.is_numeric_dtype(df[c]):
            candidate_fields.append(c)
    if candidate_fields:
        numeric_field_id = candidate_fields[0]

    if numeric_field_id:
        print(f"Using numeric field for analysis: {numeric_field_id}")
        # Outlier filtering (e.g., filter for values above a threshold)
        threshold = df[numeric_field_id].mean()  # example: filter above mean
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping by a (likely categorical) field
        group_field = None
        for c in df.columns:
            if c != numeric_field_id and df[c].dtype == object:
                group_field = c
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped (mean) by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field found in the main record set.")

## 5. Visualization
Let's plot a histogram of the numeric field and (optionally) a boxplot or bar plot by group.

> Ensure proper referencing using the `@id` of the field, as above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    # Histogram
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[main_record_set_id][numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If group_field exists, show group means
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 4))
        group_means = dataframes[main_record_set_id].groupby(group_field)[numeric_field_id].mean().sort_values()
        group_means.plot(kind='bar')
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.show()

## 6. Conclusion
This notebook introduced methods for loading, exploring, and analyzing complex Croissant datasets using `mlcroissant`. We referenced all schema entities by their `@id`, loaded data frames, performed basic cleaning, and visualized some core measures.

Key findings and next steps:
- The dataset provides structured mass spectrometry measures for extracellular vesicles from human mast cells.
- Data can be easily accessed, filtered, normalized, and grouped for downstream proteomics or biomarker discovery tasks.
- For advanced analysis, further field selection and biological annotation can be implemented using identifiers from the Croissant schema.
